In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: rebuild from the raw CSV through cleaning, the split, and every embedding, then save those results.
# False: skip that work and load the saved train, validation, and test embedding frames from data/processed/embeddings.
REBUILD = False

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","16g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split_and_weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split_and_weight(df_engineered, SEED)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)


In [ ]:
from src.embeddings import EmbeddingModel

# --- Toggle: which embeddings flow into analytics -------------------------
# Flip entries to True/False; several Trues are fine. Order = run order.
# REBUILD=True: disabled embeddings are also skipped in the predict+save loop.
# REBUILD=False: disabled embeddings are dropped right after the cache load
# (they are cached lazily, so nothing is even materialized for them).
USE_EMBEDDINGS = {"tfidf": True, "glove": False}
embedding_names = [name for name, on in USE_EMBEDDINGS.items() if on]

if REBUILD:
    embedding_model = EmbeddingModel(df_train, df_valid, df_test)
    for name, train_function in EmbeddingModel.train_registry.items():
        print(f"Embedding Model `{name}` Training")
        train_function(embedding_model)
    train_embedding_dict = {}
    valid_embedding_dict = {}
    test_embedding_dict = {}
    for name, predict_function in EmbeddingModel.predict_registry.items():
        if name not in embedding_names:
            print(f"Embedding Model `{name}` skipped (USE_EMBEDDINGS)")
            continue
        print(f"Embedding Model `{name}` Running")
        train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name] = predict_function(embedding_model)
        data_oi.save_embedding_data(
            name,
            train_embedding_dict[name],
            valid_embedding_dict[name],
            test_embedding_dict[name],
        )

else:
    train_embedding_dict, valid_embedding_dict, test_embedding_dict = data_oi.load_embedding_data()
    available = sorted(train_embedding_dict)
    missing = [n for n in embedding_names if n not in train_embedding_dict]
    if missing:
        raise FileNotFoundError(
            f"USE_EMBEDDINGS selected {missing} but the cache only has {available}. "
            "Run once with REBUILD=True to create the missing embedding cache."
        )
    for name in list(train_embedding_dict):
        if name not in embedding_names:
            del train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name]
    print(f"Embeddings in this run: {sorted(train_embedding_dict)} (cache has {available})")


In [ ]:
from src.analytics import AnalyticsModel

# --- Toggle: which analytics models run -----------------------------------
# Flip entries to True/False; several Trues are fine. Order = run order.
# Unknown keys fail loudly so a stale registry name cannot silently run nothing.
# lr_improved runs on every embedding selected above, like the SVM family.
USE_MODELS = {"lr_improved": False, "svm_l0": False, "svm_l1": False,
              "svm_tfidf_l2": True, "svm_tfidf_l3": True}
unknown = [n for n in USE_MODELS if n not in AnalyticsModel.registry]
if unknown:
    raise KeyError(f"USE_MODELS names {unknown} not in registry {sorted(AnalyticsModel.registry)}")
analytics_names = [name for name, on in USE_MODELS.items() if on]
print(f"Analytics models in this run: {analytics_names}")

analytics_models = {}
for embedding_name in train_embedding_dict.keys():
    analytics_models[embedding_name] = {
        name: AnalyticsModel(
            embedding_name,
            train_embedding_dict[embedding_name],
            valid_embedding_dict[embedding_name],
            test_embedding_dict[embedding_name],
        )
        for name in analytics_names
    }


In [ ]:
train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}

for embedding_name in train_embedding_dict.keys():
    for analytics_name in analytics_names:
        print(f"Embedding Model {embedding_name} Analytics Model {analytics_name} Running")
        model_name = f"{analytics_name}__{embedding_name}"
        train_pred_dict[model_name], valid_pred_dict[model_name], test_pred_dict[model_name] = AnalyticsModel.registry[analytics_name](
            analytics_models[embedding_name][analytics_name]
        )
        data_oi.save_prediction_data(
            model_name,
            train_pred_dict[model_name],
            valid_pred_dict[model_name],
            test_pred_dict[model_name],
        )


In [ ]:
from src.evaluation import Scorer

# --- Toggle: include the frozen baseline row (lr on raw-token baseline) ---
# The baseline LR is not in the registry: it runs on the baseline embedding
# built only in project.ipynb. Its official predictions are cached, so this
# loads `lr__baseline` from data/processed/predictions for side-by-side
# scoring instead of recomputing the whole baseline chain. Regenerate that
# cache via project.ipynb (BASELINE=True) if the upstream data ever changes.
USE_BASELINE_LR = False
if USE_BASELINE_LR:
    cached_train, cached_valid, cached_test = data_oi.load_prediction_data()
    if "lr__baseline" not in cached_train:
        raise FileNotFoundError(
            "`lr__baseline` not in data/processed/predictions; "
            "run project.ipynb once with BASELINE=True to create it."
        )
    train_pred_dict["lr__baseline"] = cached_train["lr__baseline"]
    valid_pred_dict["lr__baseline"] = cached_valid["lr__baseline"]
    test_pred_dict["lr__baseline"] = cached_test["lr__baseline"]
    print("Baseline LR row loaded from the official prediction cache.")

# Cache each prediction DataFrame before scoring: Scorer runs 5 evaluator
# jobs per model; without cache each one recomputes from the full lineage.
for d in (train_pred_dict, valid_pred_dict, test_pred_dict):
    for k, v in d.items():
        d[k] = v.cache()
        d[k].count()

scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)
